In [46]:
import json
import os
import time

import bibtexparser
import flatdict as fd
import numpy as np
import pandas as pd
import requests

from pynxtools_em.examples.oasisb_bibliography import get_bibliographical_metadata
from pynxtools_em.examples.oasisb_openalex import get_data_for_doi_from_openalex
from pynxtools_em.examples.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

/home/kaiobach/Research/hu_hu_hu/sprint_fairmat1_final_pm/pynxtools-em/examples/oasisb
/home/kaiobach/Research/paper_paper_paper/scidat_nomad_em/bb_analysis/analysis/harvest_examples/data


In [47]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype="str",
).fillna("")
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict
project_range: tuple[int, int] = (1, 880)

## Query for each project bibliographical metadata from OpenAlex

Querying metadata in addition to the DOI allows to cross-check authors, institutions, and copyright relevant details.

In [ ]:
api_queries_cnt = 0
api_queries_max = 10
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.project_name) <= project_range[1]:
            project_id = get_project_id(f"{row.project_name}")
            data_and_paper = get_bibliographical_metadata(bib, project_id)

            n_queries = get_data_for_doi_from_openalex(bib, data_and_paper)

            api_queries_cnt += n_queries  # sleep only when necessary
            if api_queries_cnt >= api_queries_max:
                sleep = float(rng.uniform(1, 10))
                print(f"Sleeping for {sleep}s")
                time.sleep(sleep)
                api_queries_cnt = 0
print("Batch querying queue done")

***

## Normalize author field cross-checking against OpenAlex

Often author names are abbreviated, e.g. Breen, A. J. instead of Breen, Andrew John.
Here we remove abbreviations.

In [ ]:
count: int = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if row.legal == "1" and row.use == "1":
            project_id = get_project_id(row.project_name)
            data_and_paper: list[str] = get_bibliographical_metadata(
                bib,
                project_id,
                verbose=False,
            )
            # print(f"{project_id}, {bib[data_and_paper[0]]['author']}")
            openalex_authors: list[str] = []
            openalex_titles: list[str] = []
            openalex_affil: int = 0
            openalex_files: list[str] = []

            # print(data_and_paper)
            for idx in [0, 1]:
                if data_and_paper[idx] != "":
                    openalex_files.append(
                        f"{os.getcwd()}{os.sep}openalex{os.sep}{data_and_paper[idx]}.json"
                    )
            for idx in [0, 1]:
                if idx < len(openalex_files):
                    if os.path.isfile(openalex_files[idx]):
                        # print(openalex_files[idx])
                        try:
                            with open(openalex_files[idx], encoding="utf-8") as fp:
                                openalex = fd.FlatDict(json.load(fp), "/")
                                for dct in openalex["authorships"]:
                                    flat = fd.FlatDict(dct, "/")
                                    if "institutions" in flat:
                                        if len(flat["institutions"]) > 0:
                                            print(flat["institutions"][0]["id"])
                                            openalex_affil = 1
                        except TypeError:
                            pass

            if openalex_affil == 0:
                count += 1
                # print(f"{project_id}, {data_and_paper[0]}")
            continue

            try:
                if os.path.isfile(openalex_files[0]):
                    with open(openalex_files[0], encoding="utf-8") as fp:
                        openalex = fd.FlatDict(json.load(fp), "/")
                        if "title" in openalex:
                            openalex_titles.append(openalex["title"].strip())
                        if "authorships" in openalex:
                            for dct in openalex["authorships"]:
                                flat = fd.FlatDict(dct, "/")
                                if "institutions" in flat:
                                    if len(flat["institutions"]) > 0:
                                        openalex_affil = 1
                                if "raw_author_name" in flat:
                                    openalex_authors.append(
                                        flat["raw_author_name"].strip()
                                    )
            except TypeError:
                pass

            # if int(project_id) in take:
            #     new_bib[data_and_paper[0]]["author"] = " and ".join(openalex_authors)
            # continue
            # 308 <= x <= 328
            # if int(project_id) < 870:
            #     continue
            # if int(project_id) > 880:
            #     break
            bibtex_authors: list[str] = []
            bibtex_titles: list[str] = []
            if data_and_paper[0] != "" and "author" in bib[data_and_paper[0]]:
                if "title" in bib[data_and_paper[0]]:
                    bibtex_titles.append(bib[data_and_paper[0]]["title"].strip())
                for bibtex_author in [
                    value.strip()
                    for value in bib[data_and_paper[0]]["author"].split("and")
                ]:
                    bibtex_authors.append(bibtex_author)

            # if len(openalex_titles) == 0 and len(bibtex_titles) == 0:
            #     #  if openalex_titles[0] != bibtex_titles[0]:
            print(
                f"{project_id}, {bib[data_and_paper[0]]['author']}"
            )  # {openalex_titles},

            """
            if openalex_authors != bibtex_authors:
                print(project_id)
                n = min(len(openalex_authors), len(bibtex_authors))
                for idx in range(0, n):
                    if openalex_authors[idx] != bibtex_authors[idx]:
                        print(f"\t{openalex_authors[idx]} != {bibtex_authors[idx]}")
                print(f"\t{openalex_authors[n:]}")
                print(f"\t{bibtex_authors[n:]}")
                print(f"{project_id}, {openalex_authors}")
                print(f"{project_id}, {bibtex_authors}")
            """
            del openalex_authors, bibtex_authors
print(f"Batch queue done {count}")

***

## Try to generate for each project a project-Name like those in APM

Get institution of first author to identify country_town_firstauthorsurname

In [22]:
import webbrowser

In [26]:
count: int = 0
query_these_institutions: set[str] = set()
project_to_institution: dict[str, str] = {}
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        if row.legal == "1" and row.use == "1":
            project_id = get_project_id(row.project_name)
            if project_range[0] <= int(project_id) <= project_range[1]:
                data_and_paper: list[str] = get_bibliographical_metadata(
                    bib,
                    project_id,
                    verbose=False,
                )
                openalex_files: list[str] = []
                for idx in [0, 1]:
                    if data_and_paper[idx] != "":
                        openalex_files.append(
                            f"{os.getcwd()}{os.sep}openalex{os.sep}{data_and_paper[idx]}.json"
                        )
                institution_ids: list[str] = []
                for idx in [0, 1]:
                    if idx < len(openalex_files):
                        if os.path.isfile(openalex_files[idx]):
                            try:
                                with open(openalex_files[idx], encoding="utf-8") as fp:
                                    openalex = fd.FlatDict(json.load(fp), "/")
                                    for dct in openalex["authorships"]:
                                        flat = fd.FlatDict(dct, "/")
                                        if "institutions" in flat:
                                            if (
                                                len(flat["institutions"]) > 0
                                                and len(institution_ids) == 0
                                            ):
                                                institution_ids.append(
                                                    flat["institutions"][0]["id"]
                                                )
                                                # if idx == 0:
                                                #     print(f"{project_id}, {institution_ids} from D")
                                                # else:
                                                #     print(f"{project_id}, {institution_ids} from A")
                            except TypeError:
                                pass
                # identify those cases for which we cannot get an automated OpenAlex institution ID, those we need to sort out by hand
                if len(institution_ids) == 0:
                    # if int(project_id) < 212:
                    #     continue
                    # print(f"{project_id}")  # , unavailable, {openalex_files}")
                    project_to_institution[project_id] = "sort_out_manually"
                else:
                    query_these_institutions.add(institution_ids[0])
                    project_to_institution[project_id] = institution_ids[0]
                    # webbrowser.open_new_tab(f"https://dx.doi.org/{bib[data_and_paper[0]]['doi']}")
                count += 1
print(f"Batch queue completed {count}")

Batch queue completed 720


In [55]:
# load manually composed list with iso3_city ids and append first author surname
draft_ods = pd.read_excel(
    f"{src_directory.replace('/bb_analysis/analysis/harvest_examples/data', '')}{os.sep}iso3_city_draft.ods",
    sheet_name="country_city_draft",
    engine="odf",
    dtype="str",
).fillna("")

nomad_project_names: dict[str, str] = {}
for row in draft_ods.itertuples(index=True):
    # if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
    data_and_paper: list[str] = get_bibliographical_metadata(
        bib,
        row.project_name,
        verbose=False,
    )
    author = (
        bib[data_and_paper[0]]["author"].split(" and ")[0].strip().split(",")
    )  # [0].strip().lower()
    project_id = get_project_id(row.project_name)
    nomad_project_names[f"{project_id}"] = (
        f"em_{row.project_name}_{row.country_city}_{author[0].lower().replace(' ', '_').replace('-', '_')}"
    )

Query automatically institution ids and compose iso3_city codes

In [29]:
import pycountry

institution_to_prefix: dict[str, str] = {}
for idx, openalex_id in enumerate(query_these_institutions):
    project_id = get_project_id(f"{idx}")
    try:
        data = requests.get(
            f"https://api.openalex.org/institutions/{openalex_id}"
        ).json()
        city = data["geo"]["city"]
        iso2 = data["geo"]["country_code"]
        iso3 = pycountry.countries.get(alpha_2=iso2).alpha_3
        prefix = f"{iso3.lower()}_{city.lower().replace(' ', '_').replace('-', '_')}"
        institution_to_prefix[openalex_id] = prefix
    except LookupError:
        institution_to_prefix[openalex_id] = "sort_out_manually"
print("Batch queries completed")

000, deu_bochum
001, kor_daejeon
002, che_zurich
003, isr_beersheba
004, deu_aachen
005, usa_williamsburg
006, swe_stockholm
007, fra_paris
008, fra_paris
009, chn_wuhan
010, chn_beijing
011, jpn_tsukuba
012, usa_northridge
013, ind_kanpur
014, pol_warsaw
015, deu_potsdam
016, can_london
017, deu_frankfurt_am_main
018, esp_oviedo
019, deu_bonn
020, jpn_tokyo
021, usa_berkeley
022, aut_graz
023, gbr_sheffield
024, zaf_bloemfontein
025, nzl_wellington
026, svn_ljubljana
027, fra_paris
028, deu_weimar
029, arg_buenos_aires
030, usa_washington
031, chn_beijing
032, usa_tempe
033, zaf_pretoria
034, fin_espoo
035, ita_turin
036, kor_seoul
037, usa_minneapolis
038, swe_gothenburg
039, cze_prague
040, deu_göttingen
041, can_edmonton
042, usa_seattle
043, fra_gif_sur_yvette
044, fra_orsay
045, nld_utrecht
046, usa_east_lansing
047, usa_philadelphia
048, cze_prague
049, usa_atlanta
050, chn_hefei
051, pol_wroclaw
052, gbr_abingdon
053, chn_xiaodian
054, esp_barcelona
055, nld_eindhoven
056, pol_

In [31]:
import yaml

with open(f"{os.getcwd()}{os.sep}institution_to_prefix.yaml", "w") as fp:
    yaml.safe_dump(institution_to_prefix, fp)

Compose all together

In [56]:
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        if row.legal == "1" and row.use == "1":
            project_id = get_project_id(row.project_name)
            if project_to_institution[project_id] == "sort_out_manually":
                if project_id in manual_nomad_project_names:
                    pass  # print(manual_nomad_project_names[project_id])
                else:
                    print(f">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>{project_id}")
            else:
                data_and_paper: list[str] = get_bibliographical_metadata(
                    bib,
                    project_id,
                    verbose=False,
                )
                author = (
                    bib[data_and_paper[0]]["author"]
                    .split(" and ")[0]
                    .strip()
                    .split(",")
                )  # [0].strip().lower()
                nomad_project_names[f"{project_id}"] = (
                    f"em_{project_id}_{institution_to_prefix[project_to_institution[project_id]]}_{author[0].lower().replace(' ', '_').replace('-', '_')}"
                )

In [58]:
with open(f"{os.getcwd()}{os.sep}aaa_nomad_project_names.yaml", "w") as fp:
    yaml.safe_dump(nomad_project_names, fp, default_style='"')

***

In [ ]:
import json

with open("apm_like_mapping.json", "w") as fp:
    json.dump(prefixes, fp, indent=4)

In [ ]:
countries: set[str] = set()
cities: set[str] = set()
for key, value in prefixes.items():
    if value.split("_", 1)[0] == "deu":
        print(f"{key}, {value}")
    countries.add(value.split("_", 1)[0])
    cities.add(value.split("_", 1)[1])
print(len(countries))
print(len(cities))

In [ ]:
openalex_ids = [
    "I108290504",
    "I193662353",
    "I139264467",
    "I2801227569",
    "I83558840",
    "I141595442",
    "I4210146223",
    "I32597200",
    "I1294671590",
    "I165339363",
    "I887968799",
    "I4210137766",
    "I80693520",
    "I114794399",
    "I154023281",
    "I130009713",
    "I19820366",
    "I4210126061",
    "I157638225",
    "I36258959",
    "I4210126723",
    "I21250087",
    "I203847022",
    "I154425047",
    "I5681781",
    "I42101026",
    "I170215575",
    "I40120149",
    "I4210091207",
    "I28407311",
    "I866009140",
    "I79510175",
    "I1336856363",
    "I161371597",
    "I167576493",
    "I1343871089",
    "I1321296531",
    "I1280414376",
    "I2802706902",
    "I17974374",
    "I98677209",
    "I4210139239",
    "I62916508",
    "I879563668",
    "I177909021",
    "I23732399",
    "I2801637956",
    "I255234318",
    "I47508984",
    "I265217849",
    "I4210095677",
    "I1288783943",
    "I27804330",
    "I171892758",
    "I152304114",
    "I4210123794",
    "I4210147556",
    "I19894307",
    "I169381384",
    "I94234084",
    "I26999989",
    "I69552723",
    "I31746571",
    "I129604602",
    "I141945490",
    "I1281735042",
    "I87216513",
    "I126345244",
    "I121980950",
    "I130238516",
    "I80281795",
    "I123044942",
    "I8087733",
    "I79576946",
    "I66958751",
    "I146655781",
    "I3124059619",
    "I2800403580",
    "I241749",
    "I2746051580",
    "I17145004",
    "I149758196",
    "I4210119393",
    "I119868032",
    "I41156924",
    "I124227911",
    "I122411786",
    "I86519309",
    "I4210107612",
    "I143910747",
    "I127251866",
    "I71999127",
    "I71999127",
    "I71999127",
    "I170979836",
    "I71999127",
    "I4210114269",
    "I201448701",
    "I74775410",
    "I4210149382",
    "I150209017",
    "I176453806",
    "I4210152878",
    "I35440088",
    "I4210106926",
    "I4210132651",
    "I904495901",
    "I2800102766",
    "I205783295",
    "I189158971",
    "I219193219",
    "I4210147295",
    "I33375025",
    "I134421475",
    "I153648349",
    "I79619799",
    "I204778367",
    "I125749732",
    "I4210114345",
    "I181647926",
    "I177605424",
    "I4210140792",
    "I183067930",
    "I1323086038",
    "I686019",
    "I4210102034",
    "I1330165540",
    "I108403487",
    "I1302918504",
    "I4210111000",
    "I163770644",
    "I102335020",
    "I31512782",
    "I123900574",
    "I112991645",
    "I1292875679",
    "I82284825",
    "I9927081",
    "I126596746",
    "I4210092374",
    "I54009628",
    "I122346577",
    "I13511017",
    "I4210144795",
    "I4210138933",
    "I190752583",
    "I4210100753",
    "I4210085980",
    "I188538660",
    "I63966007",
    "I5023651",
    "I4432739",
    "I102064193",
    "I55143463",
    "I173888879",
    "I4210165875",
    "I4387930256",
    "I4210102845",
    "I61893789",
    "I96673099",
    "I204824540",
    "I66862912",
    "I2802093357",
    "I2801997478",
    "I39586589",
    "I4210109350",
    "I181369854",
    "I25974101",
    "I197681013",
    "I4210145484",
    "I2610724",
    "I8204097",
    "I196349391",
    "I173304897",
    "I4210163238",
    "I157725225",
    "I82767444",
    "I79238269",
    "I4210136202",
    "I4210148445",
    "I881766915",
    "I2802497816",
    "I12449238",
    "I2802515739",
    "I91136226",
    "I39555362",
    "I7923278",
    "I4210164289",
    "I4210165452",
    "I4210088697",
    "I4210113422",
    "I114090438",
    "I129774422",
    "I197323543",
    "I74973139",
    "I204250578",
    "I145847075",
    "I1282311441",
    "I135598925",
    "I148283060",
    "I95457486",
    "I4210165146",
    "I11923345",
    "I75124487",
    "I1288214837",
    "I130769515",
    "I4210140723",
    "I21360634",
    "I26092322",
    "I157614274",
    "I4210091332",
    "I4210127041",
    "I187560010",
    "I155313962",
    "I4210121044",
    "I4210107764",
    "I34250744",
    "I4210118429",
    "I24354313",
    "I35928602",
    "I4210144542",
    "I4210107757",
    "I162714631",
    "I151201029",
    "I4210090776",
    "I186903577",
    "I2802711169",
    "I130828816",
    "I135310074",
    "I142974352",
    "I94624287",
    "I184942183",
    "I2801711128",
    "I119004910",
    "I2799300731",
    "I153718931",
    "I166825849",
    "I87816474",
    "I2279609970",
    "I198244214",
    "I16285277",
    "I135218257",
    "I1342911587",
    "I115228651",
    "I86501945",
    "I43980791",
    "I185261750",
    "I161593684",
    "I4210149784",
    "I4577782",
    "I154570441",
    "I4210104735",
    "I2802931824",
    "I4210097820",
    "I4210129763",
    "I180437899",
    "I4210154444",
    "I4210087453",
    "I308269434",
    "I4210157717",
    "I103187081",
    "I205640436",
    "I226560621",
    "I1280536761",
    "I205349734",
    "I165143802",
    "I1290463931",
    "I1306266525",
    "I204136569",
    "I2802613557",
    "I74801974",
    "I1289243028",
    "I75027704",
    "I47367911",
    "I2800863004",
    "I159176309",
    "I2801304276",
    "I2801876189",
    "I4210119464",
    "I4210100008",
    "I4068193",
    "I4210135359",
    "I129877168",
    "I4210091278",
    "I4387155950",
    "I3018134672",
    "I76130692",
    "I4210132079",
    "I4391767838",
    "I168879160",
    "I1334283787",
    "I4210127830",
    "I137594350",
    "I4210131793",
    "I43439940",
    "I119449181",
    "I52099693",
    "I18014758",
    "I1305919966",
    "I73613424",
    "I277688954",
    "I142476485",
    "I2799749373",
    "I71824836",
    "I70768539",
    "I2801658355",
    "I99065089",
    "I157773358",
    "I4210140493",
    "I39774598",
    "I4210126458",
    "I4210125256",
    "I4210117999",
    "I4210113496",
    "I142208455",
    "I4210164758",
    "I62396329",
    "I3122934890",
    "I4210122644",
    "I9073902",
    "I202391551",
    "I4210114611",
    "I125839683",
    "I139660479",
    "I66752286",
    "I7882870",
    "I133731052",
    "I130823665",
    "I4210119369",
    "I42766147",
    "I4210095242",
    "I51441396",
    "I202697423",
    "I4210126328",
    "I4210087815",
    "I4210165120",
    "I60134161",
    "I114457229",
    "I201537933",
    "I104946051",
    "I162015237",
    "I4210142480",
    "I4210089887",
    "I4210116620",
    "I4210111833",
    "I11935315",
    "I5124864",
    "I55732556",
    "I205401836",
    "I97018004",
    "I2801935854",
    "I158842170",
    "I4210158365",
    "I98285908",
    "I149899117",
    "I14314212",
    "I4210093216",
    "I11932220",
    "I4210140831",
    "I60587646",
    "I149213910",
    "I4210121710",
    "I131249849",
    "I69984877",
    "I4210129794",
    "I157485424",
    "I4210139925",
    "I4210151417",
    "I90610280",
    "I123387679",
    "I37796252",
    "I4210162140",
    "I22299242",
    "I32574673",
    "I142606810",
    "I189590672",
    "I65571275",
    "I184889055",
    "I39660569",
    "I4210145605",
    "I2801148539",
    "I39804081",
    "I4210094040",
    "I2699952",
    "I40034438",
    "I165368041",
    "I25217355",
    "I904437910",
    "I4210163685",
    "I4210142812",
    "I204337017",
    "I200763008",
    "I4210165664",
    "I4210128123",
    "I4210120635",
    "I72816309",
    "I22465464",
    "I4210097697",
    "I4210094421",
    "I4210088668",
    "I151295451",
    "I67327644",
    "I251424209",
    "I27837315",
    "I1282105669",
    "I4210150002",
    "I1280527723",
    "I4210103985",
    "I51057645",
    "I177877127",
    "I160993911",
    "I48615539",
    "I4210087064",
    "I96840727",
    "I4210112095",
    "I138690464",
    "I129628643",
    "I4387153566",
    "I91045830",
    "I4210155092",
    "I4210119092",
    "I4210112637",
    "I4210115143",
    "I153976015",
    "I2802229908",
    "I4210155701",
    "I65143321",
    "I30771326",
    "I2800875726",
    "I130701444",
    "I192455894",
    "I4210152560",
    "I1286593407",
    "I1297288678",
    "I1286704778",
    "I4210123595",
    "I43545212",
    "I92715842",
    "I149704539",
    "I138211613",
    "I82880672",
    "I4092182",
    "I177477856",
    "I83019370",
    "I4210102558",
    "I154099455",
    "I4210119410",
    "I9360294",
    "I4210123207",
    "I124055696",
    "I174878644",
    "I126520041",
    "I4210156502",
    "I113456305",
    "I181391015",
    "I1174212",
    "I1298590031",
    "I146399215",
    "I876193797",
    "I81556334",
    "I111112146",
    "I4210091490",
]